In [3]:
import os
import time
import pandas as pd

# --- [중요] OS 에러(SSL 인증서 경로) 방지 설정 ---
# 시스템에 잘못 설정된 인증서 경로를 무시하도록 설정합니다.
os.environ['WDM_SSL_VERIFY'] = '0' 
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
# ----------------------------------------------

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager

def crawl_paikdabang():
    # 브라우저 설정
    chrome_options = Options()
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    
    # 7. 가상 접속 보이게 설정 (Headless 모드 사용 안 함)
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    url = "https://paikdabang.com/store/"
    driver.get(url)
    
    store_data = []
    current_page = 1

    try:
        while True:
            # 5. 페이지 넘어갈 때마다 콘솔 출력
            print(f"--- 현재 {current_page} 페이지 수집 중... ---")
            
            # 1. store_list가 나타날 때까지 대기
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "store_list"))
            )
            time.sleep(1.5) # 안정적인 로딩 대기

            # BeautifulSoup 파싱
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            
            # 1. id="store_list" 안의 class="tr_list" 수집
            rows = soup.select("#store_list .tr_list")
            
            if not rows:
                print("더 이상 수집할 데이터가 없습니다.")
                break

            for row in rows:
                # 2. 클래스명에 맞춰 컬럼명 매핑
                # 지역, 매장명, 주소, 전화번호
                area = row.find(class_='td_area').get_text(strip=True) if row.find(class_='td_area') else ""
                shop = row.find(class_='td_shop').get_text(strip=True) if row.find(class_='td_shop') else ""
                addr = row.find(class_='td_addr').get_text(strip=True) if row.find(class_='td_addr') else ""
                tel = row.find(class_='td_tel').get_text(strip=True) if row.find(class_='td_tel') else ""
                
                store_data.append({
                    "지역": area,
                    "매장명": shop,
                    "주소": addr,
                    "전화번호": tel
                })

            # 3~4. 페이지네이션 처리 (다음 페이지 클릭)
            try:
                next_page_num = current_page + 1
                # li 태그 중 data-page 속성이 다음 번호인 요소를 찾음
                next_button = driver.find_elements(By.CSS_SELECTOR, f"#pagination li[data-page='{next_page_num}']")
                
                if next_button:
                    # 셀레늄 클릭이 안 먹힐 경우를 대비해 스크립트로 클릭
                    driver.execute_script("arguments[0].click();", next_button[0])
                    current_page += 1
                else:
                    print("마지막 페이지까지 수집을 완료했습니다.")
                    break
            except Exception as e:
                print(f"페이지 이동 중 알 수 없는 종료: {e}")
                break

    finally:
        # 6. CSV 저장 및 OS 에러(권한) 방지
        if store_data:
            df = pd.DataFrame(store_data)
            try:
                df.to_csv("paikdabang.csv", index=False, encoding="utf-8-sig")
                print(f"\n[성공] 총 {len(df)}개 지점 저장 완료: paikdabang.csv")
            except OSError:
                # 파일이 열려있어서 저장 안 될 경우 대비
                alt_name = f"paikdabang_{int(time.time())}.csv"
                df.to_csv(alt_name, index=False, encoding="utf-8-sig")
                print(f"\n[알림] 기존 파일이 열려있어 다른 이름으로 저장했습니다: {alt_name}")
        
        driver.quit()

if __name__ == "__main__":
    crawl_paikdabang()

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'googlechromelabs.github.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'googlechromelabs.github.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


--- 현재 1 페이지 수집 중... ---
--- 현재 2 페이지 수집 중... ---
--- 현재 3 페이지 수집 중... ---
--- 현재 4 페이지 수집 중... ---
--- 현재 5 페이지 수집 중... ---
--- 현재 6 페이지 수집 중... ---
--- 현재 7 페이지 수집 중... ---
--- 현재 8 페이지 수집 중... ---
--- 현재 9 페이지 수집 중... ---
--- 현재 10 페이지 수집 중... ---
--- 현재 11 페이지 수집 중... ---
--- 현재 12 페이지 수집 중... ---
--- 현재 13 페이지 수집 중... ---
--- 현재 14 페이지 수집 중... ---
--- 현재 15 페이지 수집 중... ---
--- 현재 16 페이지 수집 중... ---
--- 현재 17 페이지 수집 중... ---
--- 현재 18 페이지 수집 중... ---
--- 현재 19 페이지 수집 중... ---
--- 현재 20 페이지 수집 중... ---
--- 현재 21 페이지 수집 중... ---
--- 현재 22 페이지 수집 중... ---
--- 현재 23 페이지 수집 중... ---
--- 현재 24 페이지 수집 중... ---
--- 현재 25 페이지 수집 중... ---
--- 현재 26 페이지 수집 중... ---
--- 현재 27 페이지 수집 중... ---
--- 현재 28 페이지 수집 중... ---
--- 현재 29 페이지 수집 중... ---
--- 현재 30 페이지 수집 중... ---
--- 현재 31 페이지 수집 중... ---
--- 현재 32 페이지 수집 중... ---
--- 현재 33 페이지 수집 중... ---
--- 현재 34 페이지 수집 중... ---
--- 현재 35 페이지 수집 중... ---
--- 현재 36 페이지 수집 중... ---
--- 현재 37 페이지 수집 중... ---
--- 현재 38 페이지 수집 중... ---
--- 현재 39 페이지 수집 중...

In [ ]:
import pandas as pd
import requests
import time
import os
import re
from datetime import datetime
from sqlalchemy import create_engine, text
import urllib3

# [1] SSL 인증서 경로 에러 방지 및 경고 무시
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- [설정 정보] ---
KAKAO_API_KEY = "secret"
DB_USER, DB_PASS = "root", "1234"
DB_HOST, DB_PORT, DB_NAME = "localhost", "3306", "coffee_store"
CSV_FILE = "paikdabang.csv"

def clean_address(addr):
    """주소에서 지도 API가 인식하기 힘든 상세 정보 제거"""
    if not addr or pd.isna(addr): return ""
    # 괄호 및 내용 제거 (예: (전농동))
    addr = re.sub(r'\(.*?\)', '', addr)
    # 상세 층수 및 호수 제거 (예: 1층, 지하2층, 101호)
    addr = re.sub(r'\d+층|\d+호|지하\d+층|지상\d+층', '', addr)
    return " ".join(addr.split())

def get_kakao_coords(address):
    """카카오 API 호출"""
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": address}
    try:
        res = requests.get(url, headers=headers, params=params, timeout=10, verify=False)
        if res.status_code == 200:
            data = res.json()
            if data['documents']:
                return float(data['documents'][0]['y']), float(data['documents'][0]['x'])
        return None, None
    except:
        return None, None

def run_failed_data_recovery():
    if not os.path.exists(CSV_FILE):
        print(f"❌ '{CSV_FILE}' 파일이 없습니다.")
        return

    # 1. 데이터 로드
    df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()

    # '위도' 컬럼이 없으면 생성, 있으면 빈 값 확인
    if '위도' not in df.columns:
        df['위도'] = None
        df['경도'] = None

    # 2. 실패한 데이터만 필터링 (NaN 혹은 'None' 문자열 등)
    failed_mask = df['위도'].isna() | (df['위도'] == '') | (df['위도'].astype(str) == 'None')
    failed_indices = df[failed_mask].index

    if len(failed_indices) == 0:
        print("✅ 모든 데이터에 이미 좌표가 존재합니다.")
        return

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🛠️ 실패 데이터 {len(failed_indices)}건에 대해 카카오 API 보정을 시작합니다.")
    print("-" * 70)

    # 3. 보정 작업 수행
    for i in failed_indices:
        store_nm = df.at[i, '매장명']
        original_addr = df.at[i, '주소']
        
        # 주소 정제 후 API 호출
        refined_addr = clean_address(original_addr)
        lat, lon = get_kakao_coords(refined_addr)
        
        # 1차 실패 시 주소를 더 짧게 잘라서 재시도 (건물명 제외)
        if lat is None:
            short_addr = " ".join(refined_addr.split()[:4])
            lat, lon = get_kakao_coords(short_addr)

        if lat:
            df.at[i, '위도'] = lat
            df.at[i, '경도'] = lon
            status = "✅ 보정성공"
        else:
            status = "❌ 최종실패"

        now = datetime.now().strftime('%H:%M:%S')
        print(f"[{now}] ({i+1}/{len(df)}) {store_nm[:10]:<10} | {status} | 주소: {refined_addr[:25]}...")
        
        time.sleep(0.05)

    # 4. 결과 저장
    df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
    print("-" * 70)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ 보정 작업 완료 및 CSV 저장.")

if __name__ == "__main__":
    run_failed_data_recovery()

[01:08:16] 🛠️ 실패 데이터 1856건에 대해 카카오 API 보정을 시작합니다.
----------------------------------------------------------------------
[01:08:16] (1/1856) 제주애월구엄포구점( | ✅ 보정성공 | 주소: 제주특별자치도 제주시 애월읍 구엄리 606 ,...
[01:08:16] (2/1856) 오산공군기지점    | ✅ 보정성공 | 주소: 경기 평택시 신장동 229-17...
[01:08:17] (3/1856) 광명초교점      | ✅ 보정성공 | 주소: 경기도 광명시 광명로928번길 16...
[01:08:17] (4/1856) 완주혁신점      | ✅ 보정성공 | 주소: 전북특별자치도 완주군 이서면 기지로 47,...
[01:08:17] (5/1856) 자양역점       | ✅ 보정성공 | 주소: 서울 광진구 뚝섬로34길 67...
[01:08:17] (6/1856) 종로꽃시장점     | ✅ 보정성공 | 주소: 서울 종로구 종로 255...
[01:08:18] (7/1856) 목포활어회플라자점  | ✅ 보정성공 | 주소: 전남 목포시 고하대로 641-21 판매동...
[01:08:18] (8/1856) 공주동학사점     | ✅ 보정성공 | 주소: 충청남도 공주시 반포면 동학사1로 272...
[01:08:18] (9/1856) 부산일광신도시점   | ✅ 보정성공 | 주소: 부산광역시 기장군 일광읍 해빛6로 43-25...
[01:08:18] (10/1856) 부천소사본점     | ✅ 보정성공 | 주소: 경기 부천시 소사구 소사본동 82-4,...
[01:08:19] (11/1856) 빽다방 빵연구소 대 | ✅ 보정성공 | 주소: 대구 달성군 다사읍 서재리 781 ,...
[01:08:19] (12/1856) 길음뉴타운점     | ✅ 보정성공 | 주소: 서울 성북구 길음로 33 길음뉴타운제비동 제...
[01:08:19] (13/1856) 원주무극부대점  

In [2]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import os

def fill_ediya_coords(file_path):
    # 1. 데이터 로드 및 위도/경도 컬럼 준비
    if not os.path.exists(file_path):
        print(f"❌ {file_path} 파일을 찾을 수 없습니다. 경로를 확인해 주세요.")
        return

    df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # '위도'와 '경도' 컬럼이 없으면 새로 생성 (빈 값으로 초기화)
    if '위도' not in df.columns:
        df['위도'] = None
    if '경도' not in df.columns:
        df['경도'] = None
    
    # 위도 또는 경도가 없는 행만 추출 (NaN 확인)
    missing_mask = df['위도'].isna() | df['경도'].isna()
    missing_df = df[missing_mask].copy()
    
    if missing_df.empty:
        print("✅ 모든 빽다방 매장의 좌표가 이미 존재합니다.")
        return df

    print(f"📡 총 {len(missing_df)}개의 빈 좌표를 발견했습니다. 수집을 시작합니다.")

    # 2. Geopy 및 RateLimiter 설정
    # Nominatim은 무료이므로 지연 시간(min_delay_seconds)을 넉넉히(1.5초 이상) 주는 것이 중요합니다.
    geolocator = Nominatim(user_agent="ediya_korea_geocoder_v1")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

    # 3. 루프 실행 및 좌표 채우기
    start_time = time.time()
    success_count = 0
    fail_count = 0

    for idx, row in missing_df.iterrows():
        # 이디야 CSV의 주소 컬럼 사용
        address = row['주소']
        try:
            # 1단계: 전체 주소로 검색 시도
            location = geocode(address)
            
            # 2단계: 실패 시 주소를 앞 3단어(시/군/구)로 잘라서 재시도 (fallback)
            if not location:
                short_addr = " ".join(address.split()[:3])
                location = geocode(short_addr)
            
            if location:
                df.at[idx, '위도'] = location.latitude
                df.at[idx, '경도'] = location.longitude
                success_count += 1
                print(f"✔️ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 위치 확보")
            else:
                fail_count += 1
                print(f"❌ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 좌표 찾기 실패")
                
        except Exception as e:
            fail_count += 1
            print(f"❗ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 오류 발생: {e}")

    # 4. 결과 저장
    # 작업 중간에 멈춰도 데이터를 잃지 않도록 덮어쓰기 방식으로 저장합니다.
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    
    elapsed = time.time() - start_time
    print("\n" + "="*60)
    print(f"✨ 빽다방 좌표 수집 완료! (소요 시간: {elapsed:.2f}초)")
    print(f"✅ 성공: {success_count}건 | ❌ 실패: {fail_count}건")
    print(f"📂 결과가 {file_path}에 최종 업데이트되었습니다.")
    print("="*60)

    return df

# 실행 부분
if __name__ == "__main__":
    # VS Code 프로젝트 폴더 안에 ediya.csv가 있어야 합니다.
    fill_ediya_coords('paikdabang.csv')

📡 총 18개의 빈 좌표를 발견했습니다. 수집을 시작합니다.
❌ [1/18] 광명티아모IT타워점 좌표 찾기 실패
✔️ [2/18] 사천카이점 위치 확보
❌ [3/18] 빨래골입구점 좌표 찾기 실패
✔️ [4/18] 제주이도초교점 위치 확보
✔️ [5/18] 인천아인여성병원점 위치 확보
❌ [6/18] 철원와수리점 좌표 찾기 실패
❌ [7/18] 수원고색산업단지점 좌표 찾기 실패
✔️ [8/18] 구리인창점 위치 확보
❌ [9/18] 남양주2청사점 좌표 찾기 실패
✔️ [10/18] 길음롯데클라시아점 위치 확보
❌ [11/18] 범계에이스점 좌표 찾기 실패
✔️ [12/18] 여수학동e편한점 위치 확보
✔️ [13/18] 대전용운에코포레점 위치 확보
✔️ [14/18] 청주개신점 위치 확보
❌ [15/18] 대구서구청점 좌표 찾기 실패
✔️ [16/18] 무안오룡점 위치 확보
✔️ [17/18] 전주하가점 위치 확보
❌ [18/18] 안암역점 좌표 찾기 실패

✨ 빽다방 좌표 수집 완료! (소요 시간: 53.27초)
✅ 성공: 10건 | ❌ 실패: 8건
📂 결과가 paikdabang.csv에 최종 업데이트되었습니다.
